# W6D5: Document Chatbot with LangChain

## Objectives
- Build a LangChain chain using PromptTemplate, Ollama, and an output parser.
- Test the chain with 5 inputs.
- Add conversation memory and verify history across 5 turns.
- Build a two-tool agent using web search and a calculator.
- Test the agent 3 times.
**Environment:** Python virtual environment  
**LLM:** Ollama `qwen2.5:3b`


In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

print("LangChain imports successful!")

LangChain imports successful!


In [2]:
prompt = PromptTemplate.from_template(
    "You are a helpful robotics assistant. Answer the question clearly in 2-3 sentences.\n"
    "Question: {question}"
)

llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0
)

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

response = chain.invoke({
    "question": "What is the role of sensors in a robot?"
})

print(response)

Sensors in a robot play a crucial role by enabling the collection of data from the environment, which is then used to make decisions, control movements, and interact with the physical world, thereby enhancing the robot's perception and functionality.


In [3]:
questions = [
    "What is the role of sensors in a robot?",
    "What is ROS used for?",
    "How does LiDAR measure distance?",
    "What is the purpose of a PID controller?",
    "What is SLAM in robotics?"
]

for i, question in enumerate(questions, start=1):
    answer = chain.invoke({"question": question})
    print(f"Test {i}: {question}")
    print(f"Answer: {answer}\n")
    print("-" * 60)

Test 1: What is the role of sensors in a robot?
Answer: Sensors in a robot play a crucial role by enabling the collection of data from the environment, which is then used to make decisions, control movements, and interact with the physical world, thereby enhancing the robot's perception and functionality.

------------------------------------------------------------
Test 2: What is ROS used for?
Answer: ROS (Robot Operating System) is used for developing software for robots, enabling communication and collaboration among hardware components and software modules in robotic systems.

------------------------------------------------------------
Test 3: How does LiDAR measure distance?
Answer: LiDAR (Light Detection and Ranging) measures distance by emitting pulsed laser beams at objects and measuring the time it takes for the light to bounce back after hitting an object. This time measurement is then used to calculate the distance to the object.

------------------------------------------

In [4]:
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="chat_history",
    input_key="question",
    output_key="answer"
)

print("Conversation memory initialized successfully!")

Conversation memory initialized successfully!


C:\Users\ughan\AppData\Local\Temp\ipykernel_20360\3069770528.py:3: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(
c:\Users\ughan\CynarisInternship\AIML_PROJECT\venv\Lib\site-packages\pydantic\main.py:263: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [5]:
memory.save_context(
    {"question": "My favorite field is robotics."},
    {"answer": "Got it! You are interested in robotics."}
)

print(memory.load_memory_variables({})["chat_history"])

Human: My favorite field is robotics.
AI: Got it! You are interested in robotics.


In [6]:
conversation = [
    "My favorite field is robotics.",
    "I am learning ROS.",
    "I want to build an autonomous robot.",
    "Which software am I learning?",
    "What kind of robot do I want to build?"
]

for turn, question in enumerate(conversation, start=1):
    history = memory.load_memory_variables({})["chat_history"]

    turn_prompt = PromptTemplate.from_template(
        "You are a helpful robotics assistant.\n"
        "Conversation history:\n{chat_history}\n"
        "Human: {question}\n"
        "Answer briefly and use the conversation history when relevant."
    )

    turn_chain = turn_prompt | llm | StrOutputParser()

    answer = turn_chain.invoke({
        "chat_history": history,
        "question": question
    })

    memory.save_context(
        {"question": question},
        {"answer": answer}
    )

    print(f"Turn {turn}: {question}")
    print(f"AI: {answer}\n")
    print("-" * 50)

Turn 1: My favorite field is robotics.
AI: AI: That's great! Robotics is an exciting field. What specifically do you enjoy about it?

--------------------------------------------------
Turn 2: I am learning ROS.
AI: AI: That's great to hear that you are learning ROS (Robot Operating System)! ROS is a powerful framework for robotics software. What aspects of ROS are you finding most interesting or challenging?

--------------------------------------------------
Turn 3: I want to build an autonomous robot.
AI: AI: Building an autonomous robot is a fantastic goal! Learning ROS is a great start. What specific features or functionalities are you planning for your autonomous robot?

--------------------------------------------------
Turn 4: Which software am I learning?
AI: You are learning ROS (Robot Operating System). What specific features or functionalities are you planning for your autonomous robot?

--------------------------------------------------
Turn 5: What kind of robot do I want

In [7]:
history = memory.load_memory_variables({})["chat_history"]

print(history)
print("\nTotal conversation turns:", len(conversation))

Human: My favorite field is robotics.
AI: Got it! You are interested in robotics.
Human: My favorite field is robotics.
AI: AI: That's great! Robotics is an exciting field. What specifically do you enjoy about it?
Human: I am learning ROS.
AI: AI: That's great to hear that you are learning ROS (Robot Operating System)! ROS is a powerful framework for robotics software. What aspects of ROS are you finding most interesting or challenging?
Human: I want to build an autonomous robot.
AI: AI: Building an autonomous robot is a fantastic goal! Learning ROS is a great start. What specific features or functionalities are you planning for your autonomous robot?
Human: Which software am I learning?
AI: You are learning ROS (Robot Operating System). What specific features or functionalities are you planning for your autonomous robot?
Human: What kind of robot do I want to build?
AI: Based on our previous conversation, it sounds like you are focused on building an autonomous robot. What kind of tas

In [8]:
import importlib.util

packages = [
    "langchain_community",
    "ddgs",
    "duckduckgo_search",
    "langchain_classic"
]

for package in packages:
    print(f"{package}: {'Available' if importlib.util.find_spec(package) else 'Not installed'}")

langchain_community: Available
ddgs: Available
duckduckgo_search: Not installed
langchain_classic: Available


In [9]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

print("Web search tool initialized successfully!")

Web search tool initialized successfully!


C:\Users\ughan\AppData\Local\Temp\ipykernel_20360\3757714182.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [11]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Search the web for factual information."""
    return search.run(query)

@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Calculation error: {e}"

tools = [web_search, calculator]

print("Tools created:", [t.name for t in tools])

Tools created: ['web_search', 'calculator']


In [12]:
print("Calculator test:")
print(calculator.invoke({"expression": "125 * 8 + 50"}))

print("\nWeb search test:")
print(web_search.invoke({"query": "Latest stable ROS 2 distribution"}))

Calculator test:
1050

Web search test:
Rolling Distribution ROS 2 Rolling Ridley is the rolling development distribution of ROS 2. It is described in REP 2002 and was first introduced in June 2020. The Rolling distribution of ROS 2 serves two purposes: it is a staging area for future stable distributions of ROS 2, and it is a collection of the most recent development releases. As the name implies, Rolling is continuously updated ... These are the binary packages for ROS 2 Humble Hawksbill Patch Release 14 (2026-02-20) For runtime dependencies, see the binary package installation instructions. Distributions What is a Distribution? A ROS distribution is a versioned set of ROS packages. These are akin to Linux distributions (e.g. Ubuntu). The purpose of the ROS distributions is to let developers work against a relatively stable codebase until they are ready to roll everything forward. Therefore once a distribution is released, we try to limit changes to bug fixes and non-breaking ... ROS

In [13]:
from langchain_core.messages import HumanMessage, ToolMessage

agent_llm = llm.bind_tools(tools)

print("Tool-calling model initialized successfully!")

Tool-calling model initialized successfully!


In [14]:
response = agent_llm.invoke([
    HumanMessage(
        content="Use the calculator tool to calculate 125 * 8 + 50."
    )
])

print("Model response:")
print(response.content)

print("\nTool calls:")
print(response.tool_calls)

Model response:


Tool calls:
[{'name': 'calculator', 'args': {'expression': '125 * 8 + 50'}, 'id': 'a212555b-41ff-4378-a2ab-dac4f2e0ddf4', 'type': 'tool_call'}]


In [15]:
from langchain_core.messages import ToolMessage

def run_agent(question, max_steps=5):
    messages = [HumanMessage(content=question)]

    for _ in range(max_steps):
        response = agent_llm.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for tool_call in response.tool_calls:
            selected_tool = next(
                (t for t in tools if t.name == tool_call["name"]),
                None
            )

            if selected_tool is None:
                result = f"Unknown tool: {tool_call['name']}"
            else:
                result = selected_tool.invoke(tool_call["args"])

            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )

    return "Stopped after reaching the maximum tool-call steps."

print(run_agent("Use the calculator to calculate 125 * 8 + 50."))

The result of the calculation 125 * 8 + 50 is 1050.


In [16]:
agent_tests = [
    "Use the calculator to calculate 245 * 16.",
    "Use web search to find information about ROS 2 Jazzy Jalisco.",
    "Use the calculator to calculate (1500 / 25) + 175."
]

for i, question in enumerate(agent_tests, start=1):
    print(f"TEST {i}: {question}")
    try:
        answer = run_agent(question)
        print(f"Agent answer: {answer}")
    except Exception as e:
        print(f"Error: {e}")
    print("-" * 60)

TEST 1: Use the calculator to calculate 245 * 16.
Agent answer: The result of 245 * 16 is 3920.
------------------------------------------------------------
TEST 2: Use web search to find information about ROS 2 Jazzy Jalisco.
Agent answer: According to the information found, ROS 2 Jazzy Jalisco is the tenth major release of the Robot Operating System (ROS 2) ecosystem. It introduces an updated suite of software libraries, optimized middleware options, and enhanced tooling for modern robotic systems deployment. 

ROS 2, the Robot Operating System, is the software backbone behind most modern robotics projects, from research arms to warehouse robots. The official branding for the tenth generation of ROS 2 is Jazzy Jalisco.

The latest Long-Term Support (LTS) release, ROS 2 Jazzy Jalisco, was released as part of the Rolling Distribution ROS 2 Rolling Ridley. This distribution serves as a staging area for future stable distributions of ROS 2 and includes the most recent development release